In [ ]:
import wntr
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
import numpy as np
import dimod
import neal
import matplotlib.pyplot as plt
import pandas as pd
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz

filename = 'Net1.inp'
num_sensors = [3] # number of sensors to place; can be a list ex [2,4,6]

In [151]:
 # Sim data, should have 24 hour sim run
wn = wntr.network.WaterNetworkModel(filename)
sim = wntr.sim.EpanetSimulator(wn)    
results = sim.run_sim()
nodes = wn.junction_name_list
demand   = results.node['demand'][nodes]   

# Graph setup
G = nx.Graph()

for name, link in wn.links():
    L = getattr(link, 'length', 1.0)   # pumps?
    G.add_edge(link.start_node_name, link.end_node_name, length=L)

ebc_temp = nx.edge_betweenness_centrality(G, weight='length')

ebc = {}
for e, v in ebc_temp.items():
    sorted_edge = tuple(sorted(e))
    ebc[sorted_edge] = v

# assigns number index to each node
idx = {}
for i, n in enumerate(nodes):
    idx[n] = i

# Count the number of nodes, pull one with max centrality
N = len(nodes)
wmax = max(ebc.values())

w = {}

# normalizing all values
for e, v in ebc.items():
    edge = tuple(e)
    normalized_weight = v / wmax
    w[edge] = normalized_weight

In [ ]:
# just using ebc: 
# dem = []
# for n in nodes:
#     demand = wn.get_node(n).base_demand
#     dem.append(demand)

# dem = np.array(dem)

# normalized nodal costs
# highest demand node costs 0, zero demand nodes cost 1 
# optimizer will prefer busier nodes
# i think here is where we can add sim data

dem = results.node['demand'][nodes].mean().values
cost = 1 - dem / dem.max()

In [153]:
# Function takes in number of sensors (k) placement cost (cost) and budget penalty (penalty)
def build_Q(k, p_cost=0.5, penalty=5.0):
    Q = np.zeros((N, N))
    const = 0.0
    
    # uncovered-edge penalty: w(1-xi)(1-xj)
    for (a, b), wt in w.items():
        a_in, b_in = a in idx, b in idx
        if a_in and b_in:
            i, j = idx[a], idx[b]
            const += wt
            Q[i, i] -= wt; Q[j, j] -= wt
            Q[min(i,j), max(i,j)] += wt
        elif a_in or b_in:              # other end is reservoir/tank (not eligible)
            i = idx[a] if a_in else idx[b]
            const += wt; Q[i, i] -= wt

    # placement cost
    for i in range(N):
        Q[i, i] += p_cost * cost[i]
        
    # budget penalty: lam_k (sum x - k)^2
    for i in range(N):
        Q[i, i] += penalty * (1 - 2*k)
        for j in range(i+1, N):
            Q[i, j] += 2 * penalty
            
    const += penalty * k**2
    return Q, const   

def solve_gurobi(Q, const):
    m = gp.Model("sensor_qubo")
    m.Params.OutputFlag = 0

    n = Q.shape[0]
    x = m.addMVar(n, vtype=GRB.BINARY, name="x")   
    m.setObjective(x @ Q @ x + const, GRB.MINIMIZE)
                    
    t0 = time.time()
    m.optimize()
    dt = time.time() - t0

    sol = np.round(x.X).astype(int).tolist()

    nodes = convert_to_nodes(sol)
    return nodes, m.ObjVal, dt

# num reads is number of independent runs while num sweeps is how many iterations within each run
def solve_sa(Q, const, num_reads=100, num_sweeps=1000, seed=0):
    # upper-triangular matrix -> {(i, j): coeff} dict (diagonal = linear terms)
    
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q, offset=float(const))

    sampler = neal.SimulatedAnnealingSampler()
    t0 = time.time()
    sampleset = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps, seed=seed)
    dt = time.time() - t0

    best = sampleset.first  
                         
    sol = []
    for i in range(Q.shape[0]):
        sol.append(best.sample[i])
    sol = np.array(sol)   

    nodes = convert_to_nodes(sol)
    
    return nodes, best.energy, dt

# converts from bit string to actual node numbers
def convert_to_nodes(s):
    selected = []
    for n, b in zip(nodes, s):
        if b:
            selected.append(n)
    return selected

In [ ]:
# LEAK DETECTION
# sim parameters
 
duration = 6 * 3600          # 6 hours
leak_start = 3600            # Leak starts after 1 hour
timestep = 60                # update hydraulic every 60 seconds
report_timestep = 60         # update report every 60 sec
threshold = 0.5              # Minimum detectable pressure change (m)
 
leak_area = 0.0005           # Leak area (m^2)
discharge_coeff = 0.75
 
def run(leak_node=None):
    wn_sim = wntr.network.WaterNetworkModel(filename)
    wn_sim.options.time.duration = duration
    wn_sim.options.time.hydraulic_timestep = timestep
    wn_sim.options.time.report_timestep = report_timestep
    wn_sim.options.hydraulic.demand_model = 'PDD'
    if leak_node:
        wn_sim.get_node(leak_node).add_leak(wn_sim, area=leak_area, discharge_coeff=discharge_coeff, start_time=leak_start)
    
    return wntr.sim.WNTRSimulator(wn_sim).run_sim().node['pressure']

def sweep(sensors, csv_path='leak_results.csv'):
    base = run()
    rows = []
    for node in wn.junction_name_list:
        leak = run(node)
    
        # take pressure change after leak start
        pressure_change = (leak[sensors] - base[sensors]).abs()
        pressure_change_after_leak = pressure_change[pressure_change.index >= leak_start]

        # true if pressure drop exceeds threshold
        sensor_alert = pressure_change_after_leak > threshold
        any_sensor_alarming = sensor_alert.any(axis=1)

        delay = (any_sensor_alarming.idxmax() - leak_start) / 60 if any_sensor_alarming.any() else None
        rows.append({'leak_node': node, 'detect_delay_min': delay})

    df = pd.DataFrame(rows)
    #df.to_csv(csv_path, index=False)
    return df

In [155]:
for k in num_sensors:
    Q, c = build_Q(k)

    sol_g, energy_g, dt_g = solve_gurobi(Q, c)
    sol_sa, energy_sa, dt_sa = solve_sa(Q, c, num_reads=100)

    print(f"k={k}")
    print(f"   Gurobi:\n\t{sol_g}\n\tE = {energy_g:.4f}\n\ttime = {dt_g:.4f}")
    print(f"   Simulated Annealing:\n\t{sol_sa}\n\tE = {energy_sa:.4f}\n\ttime = {dt_sa:.4f}")

k=3
   Gurobi:
	['11', '12', '22']
	E = 1.7907
	time = 0.0249
   Simulated Annealing:
	['11', '12', '22']
	E = 1.7907
	time = 0.0139


In [ ]:
sweep(convert_to_nodes(sol_g), f'Net1_3sensors_gurobi.csv')

,leak_node,detect_delay_min
0,10,0.0
1,11,0.0
2,12,103.0
3,13,94.0
4,21,38.0
5,22,74.0
6,23,79.0
7,31,48.0
8,32,62.0
